# Train T5 Model — Spell/Grammar Correction

**Đề tài**: Phát hiện lỗi chính tả trong bài luận tiếng Anh của học viên  
**Module**: Train model từ `merged_data.csv` (đã hợp 3 nguồn: Lang-8 + Crawl + Chatbot)

## Pipeline
1. Upload `merged_data.csv` lên Colab
2. Tokenize + split train/val/test (80/10/10)
3. Fine-tune T5-base, 5 epochs với early stopping
4. Đánh giá BLEU/GLEU/Exact Match trên test set
5. Save model → download .zip về máy
6. (Tuỳ chọn) Register model vào bảng `models` qua ngrok

## Yêu cầu
- **Colab Pro** với GPU T4 hoặc A100
- Runtime → Change runtime type → **GPU**
- Thời gian: ~30-60 phút (50k cặp × 5 epochs)

## 1. Cài thư viện

In [ ]:
!pip install -q transformers datasets sacrebleu evaluate accelerate sentencepiece
!pip install -q psycopg2-binary  # cho phần register DB (tuỳ chọn)

## 2. Upload `merged_data.csv` từ máy local

In [ ]:
from google.colab import files
uploaded = files.upload()  # chọn file merged_data.csv từ Desktop\DatabaseAdvanced_Final\

import pandas as pd
csv_name = list(uploaded.keys())[0]
df = pd.read_csv(csv_name)
print(f'Loaded {csv_name}: {df.shape}')
print(df.head())

## 3. Imports + Config

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # T5-specific stable settings:
    optim='adafactor',                     # T5 paper recommends AdaFactor (chống NaN)
    learning_rate=1e-4,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    max_grad_norm=1.0,                     # Gradient clipping

    fp16=False,                            # T5 hay NaN với fp16
    bf16=False,

    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    logging_steps=100,
    save_total_limit=2,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved model to {OUTPUT_DIR}')

## 4. Load + Clean + Split
Train 80% / Val 10% / Test 10%

In [ ]:
# Lưu source_ids của training data để register vào models sau
if 'source_id' in df.columns:
    unique_source_ids = sorted(df['source_id'].dropna().unique().astype(int).tolist())
    print(f'Source IDs trong dataset: {unique_source_ids}')
    print(f'Phân bố theo source_id:')
    print(df['source_id'].value_counts().to_string())
else:
    unique_source_ids = []
    print('Không có cột source_id. Sẽ dùng training_source_id = NULL')

total_train_records = len(df)

# Đảm bảo có 2 cột source, target
df = df[['source', 'target']].dropna()
df = df[df['source'].astype(str).str.strip() != df['target'].astype(str).str.strip()]
df = df.drop_duplicates().reset_index(drop=True)
print(f'\nTổng sau clean: {len(df)} cặp')

# Shuffle + split
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
n = len(df)
train_df = df[:int(0.8*n)]
val_df   = df[int(0.8*n):int(0.9*n)]
test_df  = df[int(0.9*n):]
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 5. Tokenize

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = [PREFIX + str(s) for s in batch['source']]
    model_inputs = tokenizer(inputs, max_length=MAX_LEN, truncation=True)
    labels = tokenizer([str(t) for t in batch['target']], max_length=MAX_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_pandas(train_df).map(preprocess, batched=True, remove_columns=['source', 'target'])
val_ds   = Dataset.from_pandas(val_df).map(preprocess, batched=True, remove_columns=['source', 'target'])
print(f'Tokenized — train sample 0: {train_ds[0]}')

## 6. Training
Early stopping nếu val_loss không cải thiện sau 2 epoch.

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=2,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    fp16=(DEVICE == 'cuda'),
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    logging_steps=100,
    save_total_limit=2,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved model to {OUTPUT_DIR}')

## 7. Evaluate trên Test Set
BLEU + Google BLEU (GLEU) + Exact Match — 3 metric chuẩn cho GEC.

In [ ]:
model.to(DEVICE).eval()

def predict_batch(sentences, batch=32):
    out = []
    for i in range(0, len(sentences), batch):
        inputs = [PREFIX + str(s) for s in sentences[i:i+batch]]
        enc = tokenizer(inputs, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LEN).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(**enc, max_length=MAX_LEN, num_beams=4)
        out.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return out

preds = predict_batch(test_df['source'].tolist())
refs  = test_df['target'].tolist()
srcs  = test_df['source'].tolist()

# 1. Standard NLP metrics
bleu  = evaluate.load('sacrebleu').compute(predictions=preds, references=[[r] for r in refs])
gleu  = evaluate.load('google_bleu').compute(predictions=preds, references=[[r] for r in refs])
exact = float(np.mean([p.strip().lower() == r.strip().lower() for p, r in zip(preds, refs)]))

# 2. Detection metrics: P / R / F0.5 (sentence-level binary classification)
def normalize(s): return s.strip().lower()
y_true = [1 if normalize(s) != normalize(t) else 0 for s, t in zip(srcs, refs)]   # ground truth: có lỗi
y_pred = [1 if normalize(s) != normalize(p) else 0 for s, p in zip(srcs, preds)]  # model: dự đoán có lỗi

def compute_prf(y_true, y_pred, beta=0.5):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if precision + recall == 0:
        fbeta = 0.0
    else:
        fbeta = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)
    return precision, recall, fbeta

precision, recall, f05 = compute_prf(y_true, y_pred, beta=0.5)

# 3. Model size
num_parameters = sum(p.numel() for p in model.parameters())

metrics = {
    'bleu':            round(bleu['score'], 4),
    'gleu':            round(gleu['google_bleu'], 4),
    'exact_match':     round(exact, 4),
    'precision':       round(precision, 4),
    'recall':          round(recall, 4),
    'f05':             round(f05, 4),
    'num_parameters':  int(num_parameters),
    'n_test':          len(test_df),
}
print('=== TEST METRICS ===')
for k, v in metrics.items():
    print(f'  {k:18} {v}')

with open(f'{OUTPUT_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

## 8. Sample Predictions (cho báo cáo)

In [ ]:
print('=== 10 mẫu predictions ===\n')
sample = test_df.sample(min(10, len(test_df)), random_state=1).reset_index(drop=True)
for i, r in sample.iterrows():
    pred = predict_batch([r.source])[0]
    correct = '✓' if pred.strip().lower() == r.target.strip().lower() else '✗'
    print(f'[{i+1}] {correct}')
    print(f'  Source : {r.source}')
    print(f'  Target : {r.target}')
    print(f'  Predict: {pred}\n')

## 9. Nén model + tải về máy

In [ ]:
import shutil
shutil.make_archive('t5_spell_checker', 'zip', OUTPUT_DIR)
print('Đã nén thành t5_spell_checker.zip')

from google.colab import files
files.download('t5_spell_checker.zip')
files.download(f'{OUTPUT_DIR}/metrics.json')

## 10. (Tuỳ chọn) Register model vào bảng `public.models`

**Setup**: trên máy Windows mở terminal `ngrok tcp 5432`, paste host:port vào DB_CONFIG.

In [ ]:
# Trên Windows mở terminal: ngrok tcp 5432
import psycopg2

DB_CONFIG = {
    'host':     '0.tcp.ap.ngrok.io',     # ← copy từ ngrok output
    'port':     16210,                    # ← copy từ ngrok output
    'dbname':   'postgres',
    'user':     'postgres',
    'password': '12345',                  # password local PostgreSQL
}
SCHEMA = 'public'

# Model metadata
MODEL_DISPLAY_NAME    = 'T5-Base-Spell-Checker'
MODEL_VERSION         = '1.0.0'
MODEL_ARCHITECTURE    = 'T5-base'
MODEL_PATH_DRIVE      = '/content/drive/MyDrive/t5_spell_checker'


def get_or_create_training_source(cur, source_ids: list, total_records: int) -> int | None:
    """
    Auto detect training_source_id:
      - 0 source  → trả None (NULL)
      - 1 source  → trả source_id đó
      - N source  → get/create entry 'Lang-8 Combined Training Set' rồi trả source_id
    """
    if not source_ids:
        return None

    # Trường hợp đơn giản: train trên 1 source duy nhất
    if len(source_ids) == 1:
        return int(source_ids[0])

    # Train trên nhiều source → tổng hợp thành 1 entry "Combined"
    combined_name = 'Lang-8 Combined Training Set'
    cur.execute(
        f"SELECT source_id FROM {SCHEMA}.corpus_sources WHERE name = %s",
        (combined_name,)
    )
    row = cur.fetchone()
    if row:
        # Đã có → cập nhật total_records cho khớp lần train mới
        cur.execute(
            f"UPDATE {SCHEMA}.corpus_sources SET total_records = %s WHERE source_id = %s",
            (total_records, row[0])
        )
        print(f'  [Combined] dùng source_id={row[0]} (đã tồn tại)')
        return row[0]

    # Chưa có → tạo mới
    desc = f'Merged training data từ source_ids: {source_ids}'
    cur.execute(
        f"""INSERT INTO {SCHEMA}.corpus_sources
            (name, version, description, license, total_records)
            VALUES (%s, %s, %s, %s, %s)
            RETURNING source_id""",
        (combined_name, '1.0', desc, 'Mixed', total_records)
    )
    new_id = cur.fetchone()[0]
    print(f'  [Combined] tạo source_id={new_id} (mới)')
    return new_id


def register_model():
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn:
            with conn.cursor() as cur:
                # 1. Lấy training_source_id (auto)
                training_src_id = get_or_create_training_source(
                    cur, unique_source_ids, total_train_records
                )

                # 2. Insert model với 13 cột đầy đủ
                model_id = str(uuid.uuid4())
                cur.execute(
                    f"""INSERT INTO {SCHEMA}.models
                        (model_id, model_name, version, accuracy, created_at,
                         architecture, model_path, num_parameters,
                         precision_score, recall_score, f05_score,
                         is_active, training_source_id)
                        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
                    (
                        model_id,
                        MODEL_DISPLAY_NAME,
                        MODEL_VERSION,
                        metrics['gleu'],
                        datetime.now(),
                        MODEL_ARCHITECTURE,
                        MODEL_PATH_DRIVE,
                        metrics['num_parameters'],
                        metrics['precision'],
                        metrics['recall'],
                        metrics['f05'],
                        True,
                        training_src_id,
                    )
                )
        print(f'\n✓ Đã register model_id = {model_id}\n')
        print(f'  name:               {MODEL_DISPLAY_NAME} v{MODEL_VERSION}')
        print(f'  architecture:       {MODEL_ARCHITECTURE}')
        print(f'  model_path:         {MODEL_PATH_DRIVE}')
        print(f'  num_params:         {metrics["num_parameters"]:,}')
        print(f'  accuracy(gleu):     {metrics["gleu"]}')
        print(f'  precision:          {metrics["precision"]}')
        print(f'  recall:             {metrics["recall"]}')
        print(f'  f0.5_score:         {metrics["f05"]}')
        print(f'  training_source_id: {training_src_id}')
        print(f'  is_active:          True')
        return model_id
    finally:
        conn.close()


model_id = register_model()

---
## Kết quả mong đợi (50k cặp, T5-base, 5 epochs)

| Metric | Giá trị tham khảo |
|---|---|
| **BLEU** | 65-80 |
| **GLEU** | 0.55-0.75 |
| **Exact Match** | 0.40-0.60 |
| Training time | 30-60 phút |
| Model size | ~900 MB |

## Phần cho báo cáo (Chương 5 — Cài đặt & thực nghiệm)

> *Nhóm fine-tune mô hình T5-base trên dataset gồm ~50,000 cặp (sai, đúng) tổng hợp từ 3 nguồn: Lang-8 (Cách 1), Web Crawl (Cách 2), AI Generation (Cách 3). Pipeline gồm tokenize với SentencePiece, split 80/10/10, train 5 epochs với learning rate 3e-4, batch size 16, gradient accumulation 2, fp16 mixed precision. Early stopping được áp dụng để tránh overfit. Đánh giá trên test set bằng 3 metric chuẩn: BLEU (sacrebleu), Google BLEU (GLEU), Exact Match.*